# Module 8: Review Accounting Scope and Build Outcome-Specific Working Samples

目标：把 Module 7 的 42 家公司初步年度表推进为可继续研究的扩大工作样本。
仍为 8 个 Code Cells，每格先讲中文构建思路，代码、注释、文件名和输出列全部用英文。
使用普通循环、字典、条件判断和 pandas，不使用自定义财务处理函数。

本模块读取本地数据，不发送网络请求。可接在现有 Notebook 后，也可单独运行；项目目录必须与前面相同。
先完成 Module 7 Rev2，并确认已保存 656 个年度记录。
需要：candidate_annual_panel_preliminary.csv、candidate_field_sources.csv、candidate_scope_review.csv、candidate_module7_metadata.json、pilot_annual_field_sources.csv，以及原始 Company Facts 缓存。

本模块实际做四件事：
1. 对 GAP/BKE/DXLG 增补一项历史成本标签，严格匹配同一份文件和同一年度。
2. 恢复试点已保存的 9 项公司专用推导，逐项保留来源与方法。
3. 建立缺失指标待核对清单，并标记已知业务口径变化和毛利桥接差额。
4. 分别建立毛利与经营现金流年度工作样本，避免成本缺失拖累不依赖成本的现金流分析。

这里的 working sample 是可继续处理的数据工作集，不是已完成全部历史会计审核的最终样本。
新增成本标签及公司历史口径仍保留 pending。本模块不统一把所有缺失毛利填成收入减成本。
报告可以陈述数据覆盖与筛查进展；最终样本纳入、年度可比性和下一年配对仍要继续确认。

沿用完整历史期间，暂不新增统一日历年份截断。实际 period_end 不等于各公司相同的财年名称。
下一模块将在完整审核表上建立连续年度特征及下一年度结果；不能先删年份再 shift。


## Cell 1: Set Up the Project

与前面的路径相同：PROCESSED_DIR 读取整理后的 CSV，FACTS_DIR 读取原始 JSON。
只需 pandas 和 Python 自带工具，不用 SEC_HEADERS、邮箱或新的安装。
如果你改变了 Notebook 的工作目录，把 PROJECT_ROOT 设置为此前实际项目目录。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
print("Project directory:", PROJECT_ROOT)


## Cell 2: Read the Expanded Panel and Field Sources

audit 是新的工作副本，不修改 Module 7 的 CSV。
parse_dates 恢复年度日期；排序并 reset_index 方便后面用 .at[index, column] 修改指定单元格。
sources 对应每个财务字段的来源，原始金额为 USD。method 补充处理说明，原始直接标签先标为 sec_tag。

company_review 读取之前的人工工作表，cik10 保留前导零，空的人工备注保持空字符串。
公司审核状态和是否纳入的决定也读取回来，后面会尊重 exclude 决定，不自动把 pending 改成合格。
MIN_ANNUAL_ROWS=5 只控制工作样本的年度覆盖门槛，不保证年度连续或有 4 个合格配对。
原始字段缺失仍然保留 NaN；打印缺失情况是本模块处理前的基线。


In [ ]:
audit = pd.read_csv(
    PROCESSED_DIR / "candidate_annual_panel_preliminary.csv",
    parse_dates=["period_start", "period_end", "filed"],
).sort_values(["ticker", "period_end"]).reset_index(drop=True)
sources = pd.read_csv(
    PROCESSED_DIR / "candidate_field_sources.csv", parse_dates=["period_end"]
)
sources["method"] = ""
sources.loc[sources["value"].notna(), "method"] = "sec_tag"
company_review = pd.read_csv(
    PROCESSED_DIR / "candidate_scope_review.csv", dtype={"cik10": "string"}, keep_default_na=False
)
module7_metadata = json.loads(
    (PROCESSED_DIR / "candidate_module7_metadata.json").read_text(encoding="utf-8")
)
cik_lookup = company_review.set_index("ticker")["cik10"]
metrics = ["revenue", "cost_of_sales", "gross_profit", "inventory", "total_assets", "operating_cash_flow"]
MIN_ANNUAL_ROWS = 5
print("Expanded annual rows:", len(audit))
display(audit[metrics].isna().sum().rename("Missing Rows").to_frame())


## Cell 3: Recover an Additional Historical Cost Tag

在 GAP/BKE/DXLG 缓存中发现 CostOfMerchandiseSalesBuyingAndOccupancyCosts，原候选清单没有它。
它涵盖采购、占用等费用，不能看到标签名称就认定它等于别家公司选中的销售成本。
本格只对这三家公司、原成本缺失的年度尝试取值，不覆盖已取到的成本。

构建顺序：打开一家公司的缓存 → 读取这一项 USD 标签 → 遍历本公司的成本缺失年度 → 匹配 start/end/accn → 判断金额唯一。
values=list(set(values)) 去除同一金额的重复记录。如果得到两个不同金额，len(values) 不等于 1，就不填值。
再要求收入减候选成本与原报表毛利相差不超过 1 美元。这是数值对账，不是全部历史费用分类已通过审核。
两个条件都满足才写入 audit，并同步修改 sources，状态为 selected_additional_tag_pending_review。

.at 修改某个年度的金额；.loc 同步修改该年度、该文件、该指标的来源行。两张表需要保持一致。
这批缓存预计恢复 25 项成本：GAP 8、BKE 7、DXLG 10。其余年份继续缺失。

GAP 报表也说明费用分类可能与其他零售公司不同：[GAP 历史费用分类](https://www.sec.gov/Archives/edgar/data/39911/000003991119000023/R10.htm)。


In [ ]:
additional_cost_tag = "CostOfMerchandiseSalesBuyingAndOccupancyCosts"
for ticker in ["GAP", "BKE", "DXLG"]:
    path = FACTS_DIR / f"CIK{cik_lookup.loc[ticker]}_companyfacts.json"
    payload = json.loads(path.read_text(encoding="utf-8"))["payload"]
    records = payload["facts"]["us-gaap"].get(additional_cost_tag, {}).get("units", {}).get("USD", [])
    missing = audit.loc[audit["ticker"].eq(ticker) & audit["cost_of_sales"].isna()]
    for index, row in missing.iterrows():
        values = [fact["val"] for fact in records
                  if fact.get("start") == row["period_start"].strftime("%Y-%m-%d")
                  and fact["end"] == row["period_end"].strftime("%Y-%m-%d")
                  and fact["accn"] == row["accn"]]
        values = list(set(values))
        if len(values) != 1 or pd.isna(row["gross_profit"]):
            continue
        value = values[0]
        if value < 0 or abs(row["revenue"] - value - row["gross_profit"]) > 1:
            continue
        match = (sources["ticker"].eq(ticker) & sources["period_end"].eq(row["period_end"])
                 & sources["accn"].eq(row["accn"]) & sources["metric"].eq("cost_of_sales"))
        if not sources.loc[match, "status"].eq("missing").all():
            continue
        audit.at[index, "cost_of_sales"] = value
        sources.loc[match, ["tag", "value", "status", "method"]] = [additional_cost_tag, value, "selected_additional_tag_pending_review", "sec_tag"]
print("Additional cost values selected:", int(sources["status"].eq("selected_additional_tag_pending_review").sum()))


## Cell 4: Restore the Previously Checked Pilot Derivations

读取已保存的 pilot_annual_field_sources.csv，仅处理 status=derived_pending_review 的记录。
已知的试点规则是 AEO 的 7 项历史成本，以及 ANF 的 2 项毛利。不会推广到其他公司。

先匹配公司、年度终点、文件编号，再要求当前目标字段缺失。
用当前年度的输入重新计算，并与试点保存的 value 比较；输入变化或结果不一致时跳过，留给审核清单。
这样既复用前面的核对结果，也避免把来源不同或旧输入计算的数字直接复制进新表。

AEO：cost_of_sales = revenue - gross_profit。
ANF：gross_profit = revenue - cost_of_sales，但该成本排除折旧摊销，仍须保留口径说明。
来源状态改为 derived_pilot_rule_pending_review，method 保留原 derivation 文字，tag 保持空。
预计恢复 9 个字段。它们是计算值，不是新发现的直接披露标签；毛利桥接为零也不是独立验证。

试点来源：[AEO 历史成本口径](https://www.sec.gov/Archives/edgar/data/919012/000119312515087519/d848537d10k.htm)、[ANF 年报](https://www.sec.gov/Archives/edgar/data/1018840/000101884026000012/anf-20260131.htm)。


In [ ]:
pilot_sources = pd.read_csv(
    PROCESSED_DIR / "pilot_annual_field_sources.csv", parse_dates=["period_end"]
)
for index, old in pilot_sources.loc[pilot_sources["status"].eq("derived_pending_review")].iterrows():
    match = (audit["ticker"].eq(old["ticker"]) & audit["period_end"].eq(old["period_end"])
             & audit["accn"].eq(old["accn"]))
    if not match.any():
        continue
    row_index = audit.index[match][0]
    row = audit.loc[row_index]
    metric = old["metric"]
    if pd.notna(row[metric]):
        continue
    if old["ticker"] == "AEO" and metric == "cost_of_sales":
        value = row["revenue"] - row["gross_profit"]
    elif old["ticker"] == "ANF" and metric == "gross_profit":
        value = row["revenue"] - row["cost_of_sales"]
    else:
        continue
    if pd.isna(value) or value != old["value"]:
        continue
    source_match = (sources["ticker"].eq(old["ticker"]) & sources["period_end"].eq(old["period_end"])
                    & sources["accn"].eq(old["accn"]) & sources["metric"].eq(metric))
    if not sources.loc[source_match, "status"].eq("missing").all():
        continue
    audit.at[row_index, metric] = value
    sources.loc[source_match, ["value", "status", "method"]] = [value, "derived_pilot_rule_pending_review", old["derivation"]]
print("Pilot derived values restored:", int(sources["status"].eq("derived_pilot_rule_pending_review").sum()))
display(audit[metrics].isna().sum().rename("Remaining Missing Rows").to_frame())


## Cell 5: Flag Accounting Differences and Known Scope Breaks

首先记录三个已经识别的跨年度业务变化：
- AEO，period_end=2013-02-02：77kids 终止经营及比较数字重列，沿用 Module 4。
- BBWI，period_end=2022-01-29：分拆 Victoria's Secret，并在报告中把其结果列为终止经营。
- DKS，period_end=2026-01-31：2025-09-08 收购 Foot Locker，合并报表从收购日起纳入其经营结果。

这些事件使当期增长或跨期表现变化需要专门处理。代码仅打 known_scope_break 标记，年度金额保留。
下一模块会保守地排除跨越已知变化的增长或目标比较；若以后逐年核实了统一重列口径，可以有证据地修订规则。
这三个标记不是完整事件库；其他收购、业务调整、收入构成及历史重列仍需审核。
BBWI 在分拆前的业务组合也不同，不能把历史所有年度都解释为当前的美容零售业务。

gross_profit_bridge_gap=收入-成本-毛利。非零差额先标记，不自动修正。URBN 存在单独门店费用的例子。
derived_fields 统计每个年度有几个试点推导字段。推导产生的零桥接差不能视为独立证据。

base_values_ok 需要收入与资产为正、存货非负且不超过资产；缺失自然不通过这些比较。
gm_row_usable 额外需要毛利非空；ocf_row_usable 额外需要经营现金流非空。这两个判断都不要求成本存在。
负毛利、负经营现金流允许存在。比率用小数保存，0.10=10%，不乘 100 改写金额。
所有 usable 标记都仅是金额筛查，不等于已审核历史会计口径。

来源：[AEO 历史年报](https://www.sec.gov/Archives/edgar/data/919012/000119312513102293/d468081d10k.htm)、[BBWI 分拆与终止经营](https://www.sec.gov/Archives/edgar/data/701985/000070198522000009/bbwi-20220129.htm)、[DKS 收购与合并期间](https://www.sec.gov/Archives/edgar/data/1089063/000108906326000007/dks-20260131.htm)。


In [ ]:
scope_breaks = pd.DataFrame([
    {"ticker": "AEO", "period_end": "2013-02-02", "scope_reason": "Discontinued operations and comparative reclassification",
     "scope_source_url": "https://www.sec.gov/Archives/edgar/data/919012/000119312513102293/d468081d10k.htm"},
    {"ticker": "BBWI", "period_end": "2022-01-29", "scope_reason": "Victoria's Secret separation and discontinued operations",
     "scope_source_url": "https://www.sec.gov/Archives/edgar/data/701985/000070198522000009/bbwi-20220129.htm"},
    {"ticker": "DKS", "period_end": "2026-01-31", "scope_reason": "Foot Locker acquisition changes the consolidation scope",
     "scope_source_url": "https://www.sec.gov/Archives/edgar/data/1089063/000108906326000007/dks-20260131.htm"},
])
scope_breaks["period_end"] = pd.to_datetime(scope_breaks["period_end"])
audit = audit.merge(scope_breaks, on=["ticker", "period_end"], how="left", validate="one_to_one")
audit["known_scope_break"] = audit["scope_reason"].notna()
derived = sources.loc[sources["status"].eq("derived_pilot_rule_pending_review")]
derived_counts = derived.groupby(["ticker", "period_end"]).size().rename("derived_fields")
audit = audit.merge(derived_counts, on=["ticker", "period_end"], how="left")
audit["derived_fields"] = audit["derived_fields"].fillna(0).astype(int)
audit["gross_profit_bridge_gap"] = audit["revenue"] - audit["cost_of_sales"] - audit["gross_profit"]
audit["bridge_needs_review"] = audit["gross_profit_bridge_gap"].abs() > 1
audit["base_values_ok"] = (
    audit["revenue"].gt(0) & audit["total_assets"].gt(0)
    & audit["inventory"].ge(0) & audit["inventory"].le(audit["total_assets"])
)
audit["gm_row_usable"] = audit["base_values_ok"] & audit["gross_profit"].notna()
audit["ocf_row_usable"] = audit["base_values_ok"] & audit["operating_cash_flow"].notna()
audit["gross_margin"] = audit["gross_profit"] / audit["revenue"]
audit["ocf_margin"] = audit["operating_cash_flow"] / audit["revenue"]
audit.loc[~audit["gm_row_usable"], "gross_margin"] = float("nan")
audit.loc[~audit["ocf_row_usable"], "ocf_margin"] = float("nan")
display(audit.loc[audit["known_scope_break"], ["ticker", "period_end", "scope_reason"]])
display(audit.loc[audit["bridge_needs_review"], ["ticker", "period_end", "gross_profit_bridge_gap", "derived_fields"]])


## Cell 6: Build a Missing-Metric Review Queue

把仍缺失的成本和毛利取出来，合并同一年度的收入、成本、毛利，为人工核对准备上下文。
只为研究辅助给出 proposed_value：成本候选=收入-毛利，毛利候选=收入-成本。
它只写入审核清单，绝不自动写回 audit 的正式金额；另一项缺失时 proposed_value 也为空。
review_status=pending 表示尚未核对公式、报表项目与业务范围。字段状态 ambiguous 也进入清单，但不能用公式绕过金额冲突。

current_filing_url 打开本年度匹配的 SEC 文件目录。找到 10-K 正文，而不是随便打开最新年报来填旧年度。
请优先查看最新年度，再处理历史缺失，按以下顺序操作：
1. 找 Consolidated Statements of Income / Operations，确认实际年度起止及报表单位（USD、thousands 或 millions）。
2. 核对收入是否包括信用卡、租赁、其他业务收入，以及成本是否包含采购、占用、配送、折旧等。
3. 确认报表是否单列额外扣减项目，例如 URBN 门店费用。不能只因收入和成本都有值就断定毛利=两者之差。
4. 查看 Notes / Discontinued Operations / Business Combinations，检查比较年度是否重列。
5. 有充分依据后，在单独的 expanded_metric_review_decisions.csv 记录 reviewed_value、evidence_url 和 review_notes。金额填原始 USD，报表为 thousands 时乘 1,000、为 millions 时乘 1,000,000。review_status 使用 pending/approved/rejected；只有对应年度、文件、口径及金额均有证据才填 approved。本模块尚不自动应用人工填值。

示例：WSM 有报表毛利时，即便成本缺失，也可以先计算毛利率；不必为计算这个比率而填成本。
ROST/TJX 的毛利公式候选需要核实成本范围，尚未核实也不影响按已有经营现金流开展另一项结果分析。
若找不到足够证据，保留 pending 和缺失，比任意填 0 更准确。

公司层面，请在原 candidate_scope_review.csv 记录业务和口径备注；include_decision 使用 pending/include/exclude，accounting_review_status 使用 pending/reviewed。
只有核对了拟采用的历史范围并留下证据，才设置 reviewed；看一份最新年报不能代表所有历史年度已通过审核。

示例来源：[WSM 财务报表](https://www.sec.gov/Archives/edgar/data/719955/000071995526000059/wsm-20260201.htm)、[ROST 年报](https://www.sec.gov/Archives/edgar/data/745732/000074573226000006/rost-20260131.htm)。


In [ ]:
queue = sources.loc[
    sources["metric"].isin(["cost_of_sales", "gross_profit"])
    & sources["status"].isin(["missing", "ambiguous"]),
    ["ticker", "period_end", "accn", "metric", "status"],
].merge(audit[["ticker", "period_end", "accn", "period_start"] + metrics],
        on=["ticker", "period_end", "accn"], how="left", validate="many_to_one")
queue["proposed_value"] = float("nan")
cost_missing = queue["metric"].eq("cost_of_sales")
queue.loc[cost_missing, "proposed_value"] = queue["revenue"] - queue["gross_profit"]
queue.loc[~cost_missing, "proposed_value"] = queue["revenue"] - queue["cost_of_sales"]
queue.loc[cost_missing & queue["proposed_value"].lt(0), "proposed_value"] = float("nan")
queue["current_filing_url"] = (
    "https://www.sec.gov/Archives/edgar/data/"
    + queue["ticker"].map(cik_lookup).astype(int).astype(str) + "/"
    + queue["accn"].str.replace("-", "", regex=False) + "/"
)
queue["review_status"] = "pending"
queue["reviewed_value"] = float("nan")
queue["evidence_url"] = ""
queue["review_notes"] = ""
queue = queue.sort_values(["period_end", "ticker"], ascending=[False, True])
print("Cost and gross profit fields awaiting review:", len(queue))
display(queue.groupby(["ticker", "metric"]).size().rename("Review Fields").to_frame())
display(queue[["ticker", "period_end", "metric", "proposed_value", "current_filing_url"]].head(8))


## Cell 7: Create Separate Annual Working Samples

先按公司统计两个结果各自可用的年度数量，再与公司审核表合并。
这里读取 company_review 中的业务分组、CIK、公司名称和人工字段，不直接照搬 Module 7 旧的覆盖数字。
满足至少 5 个该结果可用年度、且未明确 exclude 的公司，进入对应工作样本。
pending 公司允许进入暂定工作样本，方便继续学习和配对，但不能被描述成通过最终资格审核。

毛利样本需要收入、存货、资产及毛利；现金流样本需要收入、存货、资产及经营现金流。
资产用于本项目的基础金额筛查及后续控制指标，不是计算库存增长本身的分母。
不要为了得到相同样本数量，强制将两个样本取交集。这会把毛利缺失误扩散到现金流结果，也可能改变样本选择。
以后对两种结果做直接比较时，可以额外报告共同样本敏感性分析。

按你目前这批缓存且公司决定都为 pending，预计得到：毛利工作样本 36 家、488 个年度；现金流工作样本 42 家、592 个年度。
它们是年度金额表，不是下一年度配对数，也不是 488/592 个独立公司样本。明确 exclude 或更新数据后数量会变化。

每个工作表都是“年度金额可用”表。known_scope_break 和 bridge_needs_review 保留在其中，未在这里自动删除年度。
下一模块必须从完整 audit 做上一年/下一年匹配，识别断年、财年长度变化与已知口径变化，再生成真正的配对样本。
公司最少 5 个可用年度不保证 4 个可用配对。historical review 与 point-in-time 问题也没有因这个门槛而自动解决。


In [ ]:
counts = audit.groupby("ticker").agg(
    annual_rows=("period_end", "size"), gm_usable_years=("gm_row_usable", "sum"),
    ocf_usable_years=("ocf_row_usable", "sum"), known_scope_breaks=("known_scope_break", "sum"),
    bridge_review_rows=("bridge_needs_review", "sum"), derived_fields=("derived_fields", "sum"),
)
review_columns = [
    "ticker", "subgroup", "cik10", "sec_company_name", "accounting_review_status",
    "include_decision", "business_scope_notes", "cost_and_gross_profit_notes",
]
readiness = company_review[review_columns].merge(counts, on="ticker", how="left", validate="one_to_one")
not_excluded = readiness["include_decision"].ne("exclude")
readiness["gm_working_candidate"] = readiness["gm_usable_years"].ge(MIN_ANNUAL_ROWS) & not_excluded
readiness["ocf_working_candidate"] = readiness["ocf_usable_years"].ge(MIN_ANNUAL_ROWS) & not_excluded
gm_companies = readiness.loc[readiness["gm_working_candidate"], "ticker"]
ocf_companies = readiness.loc[readiness["ocf_working_candidate"], "ticker"]
gm_working_panel = audit.loc[audit["gm_row_usable"] & audit["ticker"].isin(gm_companies)].copy()
ocf_working_panel = audit.loc[audit["ocf_row_usable"] & audit["ticker"].isin(ocf_companies)].copy()
print("Gross margin working companies:", len(gm_companies), "Annual rows:", len(gm_working_panel))
print("Operating cash flow working companies:", len(ocf_companies), "Annual rows:", len(ocf_working_panel))
display(readiness[["ticker", "gm_usable_years", "ocf_usable_years", "gm_working_candidate", "ocf_working_candidate", "include_decision"]])


## Cell 8: Save the Audit, Review Queue, and Working Samples

自动生成的 expanded_ 文件会在重跑时更新，不改写 candidate_ 或 pilot_ 原数据。
expanded_annual_audit.csv 保留全部年度与标记；expanded_field_sources.csv 保留增补标签及计算方法。
expanded_missing_metric_queue.csv 是自动生成的当前核对清单。
expanded_metric_review_decisions.csv 仅首次创建，以后保留，避免覆盖你填写的 reviewed_value、证据链接和备注。
若重跑后清单变化，下一阶段按 ticker/period_end/accn/metric 核对并应用人工决定，不能只按股票代码匹配。
本模块读取公司人工备注但不修改 candidate_scope_review.csv。

metadata 保存来源、规则和样本数。candidate_eligibility_count 仅统计你明确标为 reviewed 且 include 的公司，默认应为 0。
工作样本有几十家公司、这个人工确认计数仍为 0，并不矛盾；它们衡量的是不同阶段。
即使你以后填 reviewed，metadata 也不会自动认证所有历史会计或首次披露问题已经解决。

运行顺序：先执行全部 8 格。看 Cell 3/4 是否恢复预期字段，再看 Cell 5 的事件与桥接差额、Cell 7 的两种样本数量。
人工审核不需要这一刻一口气看完 42 家全部历史年报；先按清单优先级核对，把有证据的结论逐项记录。
如果尚未填写人工文件，可以继续运行下一模块建立暂定配对；最终报告必须说明未完成的审核及样本选择限制。


In [ ]:
outputs = {
    "expanded_annual_audit.csv": audit,
    "expanded_field_sources.csv": sources,
    "expanded_missing_metric_queue.csv": queue,
    "expanded_company_readiness.csv": readiness,
    "expanded_gross_margin_working_panel.csv": gm_working_panel,
    "expanded_ocf_working_panel.csv": ocf_working_panel,
    "expanded_known_scope_breaks.csv": scope_breaks,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
decisions_path = PROCESSED_DIR / "expanded_metric_review_decisions.csv"
if not decisions_path.exists():
    queue.to_csv(decisions_path, index=False, date_format="%Y-%m-%d")
else:
    print("Existing manual metric decisions retained:", decisions_path.name)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module7_metadata": module7_metadata,
    "annual_rows": len(audit), "unit": "USD; ratios stored as decimals",
    "additional_cost_tag": additional_cost_tag, "additional_cost_companies": ["GAP", "BKE", "DXLG"],
    "restored_pilot_derivations": len(derived), "min_annual_rows": MIN_ANNUAL_ROWS,
    "gm_working_companies": len(gm_companies), "gm_working_annual_rows": len(gm_working_panel),
    "ocf_working_companies": len(ocf_companies), "ocf_working_annual_rows": len(ocf_working_panel),
    "candidate_eligibility_count": int((readiness["accounting_review_status"].eq("reviewed") & readiness["include_decision"].eq("include")).sum()),
    "known_scope_break_rule": "Flag AEO 2013-02-02, BBWI 2022-01-29 and DKS 2026-01-31; other changes remain subject to review",
    "formula_proposals_applied": False, "manual_metric_decisions_applied": False,
    "accounting_scope_verified": False, "point_in_time_verified": False, "next_year_pairs_built": False,
}
(PROCESSED_DIR / "expanded_module8_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Module 8 outputs saved in:", PROCESSED_DIR)
